# 02 · Cutting out parking spaces ("patches")

**Goal:** turn each labeled parking space into its own small image, so a model can learn
*empty vs. occupied* one space at a time.

Decisions (and why):
- **Ignore Roboflow's train/valid/test folders.** Notebook 01 showed they mix all cameras and near-duplicate frames.
  We pool all three folders and group by **camera** instead.
- **Use a random sample of photos per camera, not all of them.** The cameras shot every 5 minutes, so neighboring
  photos are nearly identical. 1,000 photos per camera still gives ~170,000 spaces, which is plenty, and builds much faster.
- **Resize every space to 64×64 pixels.** A model needs every input to be the same size. 64×64 is enough to tell
  a car from asphalt and keeps training fast.
- **Save as image files in folders by camera and label** (`patches/PUCPR/occupied/...`). PyTorch can load folders
  laid out like this directly.
- **Save a spreadsheet (`patches/index.csv`)** with each patch's camera, label and date. Notebook 03 uses the date
  to make a fair validation set.

In [ ]:
import csv
import json
import random
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image

SPLITS = ["train", "valid", "test"]
OUT = Path("patches")
PATCH_SIZE = 64
IMAGES_PER_CAMERA = 1000
CAMERA_BY_COUNT = {100: "PUCPR", 28: "UFPR04", 40: "UFPR05"}
LABELS = {1: "empty", 2: "occupied"}

random.seed(42)  # same random sample every time you run this, so results are repeatable

## 1. Pool all photos and find each photo's camera

Same trick as notebook 01: the number of spaces in a photo tells us the camera.
Photos with no labels are skipped.

In [ ]:
photos = []  # one dict per usable photo: where it is, which camera, and its spaces

for split in SPLITS:
    with open(f"{split}/_annotations.coco.json") as f:
        coco = json.load(f)

    spaces_by_image = {}
    for a in coco["annotations"]:
        spaces_by_image.setdefault(a["image_id"], []).append(a)

    for img in coco["images"]:
        spaces = spaces_by_image.get(img["id"], [])
        camera = CAMERA_BY_COUNT.get(len(spaces))
        if camera is None:
            continue  # no labels (or an unexpected count), so we can't use it
        photos.append({
            "path": f"{split}/{img['file_name']}",
            "camera": camera,
            "date": img["file_name"][:10],  # file names start with the capture date, e.g. 2012-09-14
            "spaces": spaces,
        })

print("Usable photos per camera:", Counter(p["camera"] for p in photos))

## 2. Pick a random sample per camera

In [ ]:
sample = []
for camera in ["PUCPR", "UFPR04", "UFPR05"]:
    camera_photos = [p for p in photos if p["camera"] == camera]
    sample += random.sample(camera_photos, min(IMAGES_PER_CAMERA, len(camera_photos)))

print("Photos in sample:", Counter(p["camera"] for p in sample))
print("Spaces that will become patches:", sum(len(p["spaces"]) for p in sample))

## 3. Cut out every space and save it

For each photo: open it once, then for each space crop its box `[x, y, w, h]`,
resize to 64×64, and save it into `patches/<camera>/<label>/`.

This writes ~170,000 small files, so expect it to take a few minutes.

In [ ]:
for camera in ["PUCPR", "UFPR04", "UFPR05"]:
    for label in LABELS.values():
        (OUT / camera / label).mkdir(parents=True, exist_ok=True)

rows = []
for n, photo in enumerate(sample):
    image = Image.open(photo["path"]).convert("RGB")
    stem = Path(photo["path"]).stem[:19]  # e.g. 2012-09-14_09_06_10 (date and time)

    for space in photo["spaces"]:
        x, y, w, h = space["bbox"]
        # crop takes (left, top, right, bottom), so convert from (x, y, width, height)
        patch = image.crop((x, y, x + w, y + h)).resize((PATCH_SIZE, PATCH_SIZE))

        label = LABELS[space["category_id"]]
        file = OUT / photo["camera"] / label / f"{stem}_{space['id']}.jpg"
        patch.save(file, quality=95)
        rows.append({"file": file.as_posix(), "camera": photo["camera"],
                     "label": label, "date": photo["date"]})

    if (n + 1) % 500 == 0:
        print(f"  {n + 1}/{len(sample)} photos done")

with open(OUT / "index.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["file", "camera", "label", "date"])
    writer.writeheader()
    writer.writerows(rows)

print("Saved", len(rows), "patches")

## 4. Check the result

Always look at your data after a processing step, since bugs here silently ruin everything later.

In [ ]:
counts = Counter((r["camera"], r["label"]) for r in rows)
for camera in ["PUCPR", "UFPR04", "UFPR05"]:
    e, o = counts[(camera, "empty")], counts[(camera, "occupied")]
    print(f"{camera:7s} empty: {e:6d}   occupied: {o:6d}   ({o / (e + o):.0%} occupied)")

print()
print("Days of footage per camera:")
for camera in ["PUCPR", "UFPR04", "UFPR05"]:
    days = {r["date"] for r in rows if r["camera"] == camera}
    print(f"  {camera:7s} {len(days)} days ({min(days)} to {max(days)})")

In [ ]:
# 8 random empty and 8 random occupied patches from each camera
fig, axes = plt.subplots(6, 8, figsize=(12, 9))
for ax in axes.flat:
    ax.axis("off")
row_index = 0
for camera in ["PUCPR", "UFPR04", "UFPR05"]:
    for label in ["empty", "occupied"]:
        group = [r for r in rows if r["camera"] == camera and r["label"] == label]
        picks = random.sample(group, min(8, len(group)))
        for col, r in enumerate(picks):
            ax = axes[row_index, col]
            ax.imshow(Image.open(r["file"]))
            ax.axis("off")
        axes[row_index, 0].set_title(f"{camera} {label}", loc="left", fontsize=9)
        row_index += 1
plt.tight_layout()
plt.show()

## What to look for in the grid

- Empty patches should mostly show asphalt / painted lines; occupied should show a car.
- Some empty patches will show **part of a neighboring car**. That's the box-overlap problem from notebook 01.
  These are the hard cases the model will struggle with, and they're worth mentioning in your README.
- PUCPR patches look different from UFPR ones (camera angle, distance). That difference is exactly what
  makes "train on UFPR, test on PUCPR" a real test.

**Next (notebook 03):** train a small neural network on the UFPR patches and test it on PUCPR.